In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import time, os, gc, re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList

# ==========================================
# CẤU HÌNH
# ==========================================
MODEL_A_PATH = "/kaggle/input/datasets/ductaiphan/nanoreason-3b/NanoReason-3B-Final"
MODEL_B_PATH = "Qwen/Qwen2.5-3B-Instruct"
MAX_NEW_TOKENS = 512

history = []

# ==========================================
# CẢM BIẾN PHANH — chỉ dừng khi \boxed{} hoàn chỉnh
# ==========================================
class StopOnBoxed(StoppingCriteria):
    def __init__(self, tokenizer, input_len):
        self.tokenizer = tokenizer
        self.input_len = input_len
        self._stopped  = False

    def __call__(self, input_ids, scores, **kwargs):
        if self._stopped:
            return True
        new_ids = input_ids[0][self.input_len:]
        text    = self.tokenizer.decode(new_ids, skip_special_tokens=True)
        if "\\boxed{" in text:
            start  = text.find("\\boxed{")
            braces = 0
            for i in range(start + 6, len(text)):
                if   text[i] == '{': braces += 1
                elif text[i] == '}':
                    braces -= 1
                    if braces == 0:
                        self._stopped = True
                        return True
        return False

# ==========================================
# HELPER: load / unload
# ==========================================
def load_model(path):
    print(f"  📦 Đang load: {path} ...")
    tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        path,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="cuda",
    )
    model.eval()
    vram = torch.cuda.memory_allocated() / 1024**3
    print(f"  ✅ Load xong. VRAM dùng: {vram:.2f} GB")
    return tokenizer, model

def unload_model(tokenizer, model):
    del tokenizer, model
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(1)

# ==========================================
# FORMATTER
# Xử lý 2 trường hợp:
#   A) Có \boxed{}     → lấy box cuối, xóa "Final Answer" thừa phía trước
#   B) Không có \boxed{} → tìm "Final Answer: X", chỉ giữ lần đầu tiên
# ==========================================
def clean_output(text: str) -> str:

    # ── Trường hợp A: có \boxed{} ──
    all_boxes    = []
    search_start = 0
    while True:
        pos = text.find("\\boxed{", search_start)
        if pos == -1:
            break
        braces, end = 0, -1
        for i in range(pos + 6, len(text)):
            if   text[i] == '{': braces += 1
            elif text[i] == '}':
                braces -= 1
                if braces == 0:
                    end = i
                    break
        if end != -1:
            all_boxes.append((pos, end))
            search_start = end + 1
        else:
            break

    if all_boxes:
        last_start, last_end = all_boxes[-1]
        boxed_str   = text[last_start : last_end + 1]
        text_before = text[:all_boxes[0][0]].strip()
        # Xóa mọi dòng "Final Answer..." thừa trước box
        text_before = re.sub(
            r'(?im)^\s*(final\s*answer|the\s*answer\s*is)[\s:=\-]*.*$',
            '', text_before
        ).strip()
        return f"{text_before}\n\n**Final Answer:** {boxed_str}"

    # ── Trường hợp B: plain text, không có \boxed{} ──
    fa_pattern = re.compile(r'(?i)final\s*answer\s*[:\-=]?\s*')
    matches     = list(fa_pattern.finditer(text))

    if not matches:
        return text.strip()

    # Lấy lần xuất hiện đầu tiên, cắt bỏ mọi thứ sau (chống spam)
    first      = matches[0]
    after      = text[first.end():]
    answer_val = after.split('\n')[0].strip()
    text_before = text[:first.start()].strip()

    return f"{text_before}\n\n**Final Answer:** {answer_val}"

# ==========================================
# HELPER: generate
# ==========================================
def get_eos_ids(tokenizer):
    eos_ids = [tokenizer.eos_token_id]
    try:
        im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
        if im_end_id != tokenizer.unk_token_id and im_end_id not in eos_ids:
            eos_ids.append(im_end_id)
    except Exception:
        pass
    return eos_ids

def generate(tokenizer, model, prompt):
    inputs    = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_len = inputs["input_ids"].shape[1]

    stopping_criteria = StoppingCriteriaList([StopOnBoxed(tokenizer, input_len)])
    eos_ids = get_eos_ids(tokenizer)

    t0 = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            input_ids          = inputs["input_ids"],
            attention_mask     = inputs["attention_mask"],
            max_new_tokens     = MAX_NEW_TOKENS,
            temperature        = 0.1,
            do_sample          = True,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.eos_token_id,
            eos_token_id       = eos_ids,
            stopping_criteria  = stopping_criteria,
        )
    elapsed = time.time() - t0

    new_ids = output_ids[0][input_len:]
    text    = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    text    = clean_output(text)
    tokens  = len(new_ids)
    tps     = tokens / max(elapsed, 0.001)

    return text, tokens, elapsed, tps

# ==========================================
# HÀM SO SÁNH CHÍNH
# ==========================================
def compare_models(question):
    if not question.strip():
        return "⚠️ Vui lòng nhập câu hỏi!", "", ""

    configs = [
        (
            "NanoReason-3B",
            MODEL_A_PATH,
            # FIX: KHÔNG mớm "#### UNDERSTAND ####" vào prompt.
            # Để model tự sinh toàn bộ từ đầu, bao gồm cả section UNDERSTAND.
            # Nếu mớm header vào input thì new_ids sẽ không chứa phần UNDERSTAND.
            f"Question: {question}\n\nSolution:\n",
        ),
        (
            "Qwen2.5-3B-Instruct",
            MODEL_B_PATH,
            f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n",
        ),
    ]

    results = {}
    for label, path, prompt in configs:
        print(f"\n{'='*50}")
        print(f"🔄 [{label}] Bắt đầu...")
        try:
            tokenizer, model = load_model(path)
            print(f"  ⚡ Đang sinh lời giải...")
            text, tokens, elapsed, tps = generate(tokenizer, model, prompt)
            results[label] = dict(text=text, tokens=tokens, elapsed=elapsed, tps=tps, error=None)
            print(f"  📊 {tokens} tokens | {elapsed:.2f}s | {tps:.1f} tok/s")
        except Exception as e:
            print(f"  ❌ Lỗi: {e}")
            results[label] = dict(text=f"❌ Lỗi: {e}", tokens=0, elapsed=0, tps=0, error=str(e))
        finally:
            try: unload_model(tokenizer, model)
            except: pass

    a = results["NanoReason-3B"]
    b = results["Qwen2.5-3B-Instruct"]

    history.append({
        "q":     question[:55] + ("..." if len(question) > 55 else ""),
        "tps_a": round(a["tps"], 1),     "tps_b": round(b["tps"], 1),
        "tok_a": a["tokens"],            "tok_b": b["tokens"],
        "t_a":   round(a["elapsed"], 2), "t_b":   round(b["elapsed"], 2),
    })

    def fmt(label, r):
        header = f"⚡ {r['tps']:.1f} tok/s  |  🕐 {r['elapsed']:.2f}s  |  📝 {r['tokens']} tokens"
        sep    = "─" * 50
        return f"{header}\n{sep}\n{r['text']}"

    w_speed  = "🏆 NanoReason" if a["tps"]     > b["tps"]     else "🏆 Qwen Gốc"
    w_time   = "🏆 NanoReason" if a["elapsed"] < b["elapsed"] else "🏆 Qwen Gốc"
    w_length = "🏆 NanoReason" if a["tokens"]  > b["tokens"]  else "🏆 Qwen Gốc"

    nano_wins = sum([a["tps"] > b["tps"], a["elapsed"] < b["elapsed"], a["tokens"] > b["tokens"]])
    verdict   = "🎉 NanoReason-3B thắng tổng thể!" if nano_wins >= 2 else "🎉 Qwen Gốc thắng tổng thể!"

    summary = (
        f"{'Tiêu chí':<22} {'NanoReason':>13} {'Qwen Gốc':>13} {'Thắng':>15}\n"
        f"{'─'*66}\n"
        f"{'Tốc độ (tok/s)':<22} {a['tps']:>13.1f} {b['tps']:>13.1f} {w_speed:>15}\n"
        f"{'Thời gian (s)':<22} {a['elapsed']:>13.2f} {b['elapsed']:>13.2f} {w_time:>15}\n"
        f"{'Số tokens sinh':<22} {a['tokens']:>13} {b['tokens']:>13} {w_length:>15}\n"
        f"{'─'*66}\n{verdict}\n\n📊 Tổng câu đã so sánh: {len(history)}"
    )

    return fmt("NanoReason-3B", a), fmt("Qwen2.5-3B-Instruct", b), summary

def get_history():
    if not history: return "Chưa có dữ liệu. Hãy chạy ít nhất 1 câu hỏi!"
    lines = [
        f"{'#':<3} {'Câu hỏi':<42} {'TPS_A':>6} {'TPS_B':>6} {'Tok_A':>6} {'Tok_B':>6} {'T_A':>6} {'T_B':>6}",
        "─" * 88
    ]
    for i, h in enumerate(history, 1):
        lines.append(
            f"{i:<3} {h['q']:<42} {h['tps_a']:>6} {h['tps_b']:>6} "
            f"{h['tok_a']:>6} {h['tok_b']:>6} {h['t_a']:>6} {h['t_b']:>6}"
        )
    if len(history) > 1:
        avg_tps_a = sum(x["tps_a"] for x in history) / len(history)
        avg_tps_b = sum(x["tps_b"] for x in history) / len(history)
        avg_t_a   = sum(x["t_a"]   for x in history) / len(history)
        avg_t_b   = sum(x["t_b"]   for x in history) / len(history)
        sum_tok_a = sum(x["tok_a"] for x in history)
        sum_tok_b = sum(x["tok_b"] for x in history)
        lines.extend([
            "─" * 88,
            f"{'TB':<3} {'':<42} {avg_tps_a:>6.1f} {avg_tps_b:>6.1f} "
            f"{sum_tok_a:>6} {sum_tok_b:>6} {avg_t_a:>6.1f} {avg_t_b:>6.1f}"
        ])
        lines.append(
            f"\n🏆 NanoReason tổng thể tối ưu hơn!"
            if avg_tps_a > avg_tps_b else
            f"\n🏆 Qwen Gốc tổng thể tối ưu hơn!"
        )
    return "\n".join(lines)

def clear_history():
    global history
    history = []
    return "🗑️ Đã xóa lịch sử."

# ==========================================
# GIAO DIỆN GRADIO
# ==========================================
EXAMPLES = [
    ["Janet's ducks lay 16 eggs per day. She eats 3 for breakfast and bakes muffins with 4. She sells the rest at $2/egg. How much does she make daily?"],
    ["There are 5 red balls and 3 blue balls in a bag. What is the probability of drawing a red ball?"],
]

with gr.Blocks(theme=gr.themes.Soft(), title="NanoReason vs Qwen") as demo:
    gr.Markdown(
        "# ⚡ So Sánh NanoReason-3B vs Qwen2.5-3B-Instruct\n"
        "> 🛑 Đã trang bị Auto-Stop Cảm Biến & Bulldozer Formatter!\n"
        "> ✅ **Fix:** Model tự sinh toàn bộ từ UNDERSTAND — không mớm header vào prompt nữa."
    )

    with gr.Tab("🔬 So Sánh Trực Tiếp"):
        txt_input = gr.Textbox(lines=4, label="📝 Nhập câu hỏi / đề toán")
        with gr.Row():
            btn_compare     = gr.Button("🚀 So sánh 2 mô hình", variant="primary",   scale=3)
            btn_clear_input = gr.Button("🗑️ Xóa",               variant="secondary", scale=1)
        with gr.Row():
            with gr.Column():
                gr.Markdown("### 🤖 NanoReason-3B *(Của bạn)*")
                out_a = gr.Textbox(lines=16, label="Kết quả", show_copy_button=True, interactive=False)
            with gr.Column():
                gr.Markdown("### 🏛️ Qwen2.5-3B-Instruct *(Gốc)*")
                out_b = gr.Textbox(lines=16, label="Kết quả", show_copy_button=True, interactive=False)
        out_summary = gr.Textbox(lines=9, label="Bảng So Sánh", interactive=False)
        gr.Examples(examples=EXAMPLES, inputs=txt_input)

        btn_compare.click(fn=compare_models, inputs=txt_input, outputs=[out_a, out_b, out_summary])
        btn_clear_input.click(fn=lambda: ("", "", "", ""), outputs=[txt_input, out_a, out_b, out_summary])

    with gr.Tab("📈 Lịch Sử Benchmark"):
        with gr.Row():
            btn_hist    = gr.Button("🔄 Cập nhật lịch sử", variant="primary")
            btn_clrhist = gr.Button("🗑️ Xóa lịch sử",      variant="stop")
        out_hist = gr.Textbox(lines=22, label="Lịch sử benchmark", interactive=False)
        btn_hist.click(fn=get_history,      outputs=out_hist)
        btn_clrhist.click(fn=clear_history, outputs=out_hist)

demo.launch(share=True)